# Versión reproducible - FASE3

Proyecto Bike Sharing Analytics.

Adaptación del notebook académico original.

Se utiliza el dataset local data/day.csv.
Los resultados nuevos se guardan por separado.

Instalar previamente requirements.txt.

Los algoritmos y parámetros originales se conservan.

La ejecución integral está pendiente de validación.

Notebook de origen:
notebooks/evidencias/Fases_2_3/fase 2 y fase 3/fase 3/Fase_3_Modelado_Avanzado_Bicicletas.ipynb

In [ ]:

# Localizar la raíz del repositorio.

from pathlib import Path
import sys
import hashlib

actual = Path.cwd().resolve()

candidatos = [actual, *actual.parents]

# Alternativa para Google Colab.

candidatos.append(
    Path("/content/Bike-Sharing-Analytics")
)

ROOT = next(
    (
        p for p in candidatos
        if (p / "data" / "day.csv").is_file()
        and (p / "requirements.txt").is_file()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Descarga o clona el repositorio antes de ejecutar."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.config import DATASET, RESULTS_DIR

HASH_ESPERADO = (
    "a6bcf826782d3c0fbfdcbeead17cd0884"
    "185a0dafe8ff10cd48a874ee7ba18be"
)

assert hashlib.sha256(
    DATASET.read_bytes()
).hexdigest() == HASH_ESPERADO

print("Repositorio:", ROOT)
print("Dataset:", DATASET)
print("Integridad del dataset: OK")



# FASE 3 — Modelado Avanzado, Clustering, Optimización e Interpretabilidad

**Proyecto Integrador — Ciencia de Datos**  
**Universidad de La Salle**  
**Dataset:** Bike Sharing Dataset — day.csv

## Objetivo

Desarrollar y comparar modelos supervisados y no supervisados para analizar la demanda diaria de bicicletas compartidas, aplicando técnicas de regresión regularizada, algoritmos basados en árboles, clasificación y clustering.

## Algoritmos

1. Regresión: Ridge y Random Forest Regressor.
2. Clasificación: KNN y Random Forest Classifier.
3. Clustering: K-Means y Agglomerative Clustering.

## Metodología

Se conservará una partición cronológica de entrenamiento (70 %), validación (15 %) y prueba (15 %). El preprocesamiento y la optimización se ajustarán sin utilizar información del conjunto de prueba.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 2
# Importación de librerías y carga del dataset
# ==========================================================

from pathlib import Path
import urllib.request
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibilidad
SEMILLA = 42
np.random.seed(SEMILLA)

# Carpetas del proyecto
BASE = RESULTS_DIR / "reproduccion" / "fase3"
DATOS = BASE / "datos"
RESULTADOS = BASE / "resultados"

DATOS.mkdir(parents=True, exist_ok=True)
RESULTADOS.mkdir(parents=True, exist_ok=True)

# Dataset local del repositorio
archivo_csv = DATASET

# Cargar datos
df = pd.read_csv(archivo_csv)

print("Dataset cargado correctamente")
print("Dimensiones:", df.shape)
print("Columnas:", df.columns.tolist())

display(df.head())


In [ ]:

# ==========================================================
# FASE 3 - CELDA 3
# Validación y control de calidad
# ==========================================================

# Copia de trabajo para no modificar los datos originales
df_modelo = df.copy()

# Convertir fecha y ordenar cronológicamente
df_modelo["dteday"] = pd.to_datetime(
    df_modelo["dteday"]
)

df_modelo = (
    df_modelo
    .sort_values("dteday")
    .reset_index(drop=True)
)

# Diagnóstico de calidad
print("Dimensiones:", df_modelo.shape)
print("Valores faltantes:", df_modelo.isna().sum().sum())
print("Fechas duplicadas:", df_modelo["dteday"].duplicated().sum())

print(
    "Periodo:",
    df_modelo["dteday"].min().date(),
    "a",
    df_modelo["dteday"].max().date()
)

print(
    "Promedio de alquileres:",
    round(df_modelo["cnt"].mean(), 2)
)

# Validaciones automáticas
assert df_modelo.shape == (731, 16)
assert df_modelo.isna().sum().sum() == 0
assert df_modelo["dteday"].duplicated().sum() == 0
assert (df_modelo["cnt"] == (
    df_modelo["casual"] + df_modelo["registered"]
)).all()

print("Validación de calidad superada.")


In [ ]:

# ==========================================================
# FASE 3 - CELDA 4
# División cronológica 70 % / 15 % / 15 %
# ==========================================================

n = len(df_modelo)

n_train = int(n * 0.70)
n_val = int(n * 0.15)

train_df = df_modelo.iloc[:n_train].copy()

val_df = df_modelo.iloc[
    n_train:n_train + n_val
].copy()

test_df = df_modelo.iloc[
    n_train + n_val:
].copy()

for nombre, datos in [
    ("Entrenamiento", train_df),
    ("Validación", val_df),
    ("Prueba", test_df)
]:
    print(
        f"{nombre}: {len(datos)} registros | "
        f"{datos['dteday'].min().date()} a "
        f"{datos['dteday'].max().date()}"
    )

# Verificar que no existan fechas compartidas
assert train_df["dteday"].max() < val_df["dteday"].min()
assert val_df["dteday"].max() < test_df["dteday"].min()

assert (
    len(train_df) + len(val_df) + len(test_df)
) == len(df_modelo)

print("División cronológica correcta.")



## 3. Ingeniería de características

Se construyen dos variables derivadas para enriquecer la información disponible:

- **fin_de_semana:** identifica los sábados y domingos, considerando que los patrones de movilidad pueden cambiar respecto a los días laborales.
- **temp_hum_interaccion:** representa la interacción entre temperatura y humedad, con el propósito de capturar su posible efecto conjunto sobre la demanda.

Ambas variables se calculan exclusivamente a partir de características disponibles antes de conocer la cantidad de alquileres. No se utilizan `casual` ni `registered` como predictores, dado que su suma corresponde directamente a `cnt`, lo que produciría fuga de información.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 5
# Ingeniería de características
# ==========================================================

def crear_caracteristicas(datos):
    """
    Crea variables derivadas sin utilizar la variable objetivo.
    """
    resultado = datos.copy()

    # 1. Identificar sábados y domingos
    # En pandas: lunes=0, domingo=6
    resultado["fin_de_semana"] = (
        resultado["dteday"].dt.dayofweek >= 5
    ).astype(int)

    # 2. Interacción entre temperatura y humedad
    resultado["temp_hum_interaccion"] = (
        resultado["temp"] * resultado["hum"]
    )

    return resultado


# Aplicar la misma transformación a cada conjunto
train_df = crear_caracteristicas(train_df)
val_df = crear_caracteristicas(val_df)
test_df = crear_caracteristicas(test_df)

# Verificar nuevas columnas
columnas_nuevas = [
    "fin_de_semana",
    "temp_hum_interaccion"
]

for nombre, datos in [
    ("Entrenamiento", train_df),
    ("Validación", val_df),
    ("Prueba", test_df)
]:
    assert all(
        columna in datos.columns
        for columna in columnas_nuevas
    )

    assert datos[columnas_nuevas].isna().sum().sum() == 0

    print(
        f"{nombre}: {datos.shape[0]} filas, "
        f"{datos.shape[1]} columnas"
    )

print("\nEjemplo de las nuevas características:")

display(
    train_df[
        [
            "dteday",
            "temp",
            "hum",
            "fin_de_semana",
            "temp_hum_interaccion",
            "cnt"
        ]
    ].head(10)
)

print("\nIngeniería de características completada.")



## 4. Selección de variables y preprocesamiento

La variable objetivo es `cnt`, que representa la cantidad diaria de alquileres.

Se seleccionan variables meteorológicas y de calendario para predecir la demanda.

Se excluyen `casual` y `registered`, porque su suma corresponde a `cnt`, lo que ocasionaría fuga de información.

Las variables numéricas se estandarizan mediante `StandardScaler`, mientras que las categóricas se transforman mediante `OneHotEncoder`.

El preprocesamiento se ajusta exclusivamente con los datos de entrenamiento y luego se aplica a validación y prueba.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 6
# Selección de variables y preprocesamiento
# ==========================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Variables predictoras
numericas = [
    "temp",
    "hum",
    "windspeed",
    "temp_hum_interaccion"
]

categoricas = [
    "season",
    "weathersit",
    "workingday",
    "yr",
    "fin_de_semana"
]

predictores = numericas + categoricas

# Separar variables explicativas y objetivo
X_train = train_df[predictores].copy()
X_val = val_df[predictores].copy()
X_test = test_df[predictores].copy()

y_train = train_df["cnt"].copy()
y_val = val_df["cnt"].copy()
y_test = test_df["cnt"].copy()

# Transformación numérica y categórica
preprocesador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            StandardScaler(),
            numericas
        ),
        (
            "categoricas",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categoricas
        )
    ]
)

# Ajustar SOLO con entrenamiento
X_train_procesado = preprocesador.fit_transform(X_train)

# Aplicar transformaciones aprendidas
X_val_procesado = preprocesador.transform(X_val)
X_test_procesado = preprocesador.transform(X_test)

print("Entrenamiento:", X_train_procesado.shape)
print("Validación:", X_val_procesado.shape)
print("Prueba:", X_test_procesado.shape)

# Verificaciones
assert X_train_procesado.shape[0] == 511
assert X_val_procesado.shape[0] == 109
assert X_test_procesado.shape[0] == 111

assert np.isfinite(X_train_procesado).all()
assert np.isfinite(X_val_procesado).all()
assert np.isfinite(X_test_procesado).all()

print("Preprocesamiento completado sin fuga de información.")



## 5. Modelado avanzado: regresión Ridge

Se implementa Ridge, una regresión lineal multivariable con regularización L2.

La regularización penaliza los coeficientes elevados para controlar la complejidad del modelo.

Se evalúan diferentes valores de `alpha` utilizando exclusivamente el conjunto de validación. El mejor modelo se selecciona mediante RMSE.

**Métricas utilizadas:**
- MAE: error absoluto medio.
- RMSE: raíz del error cuadrático medio.
- R²: coeficiente de determinación.

El conjunto de prueba se reserva para la evaluación final, después de comparar los modelos candidatos.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 7
# Entrenamiento y optimización de Ridge
# ==========================================================

from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Valores de regularización que vamos a comparar
alphas = [0.01, 0.1, 1, 10, 100, 1000]

resultados_ridge = []
modelos_ridge = {}

for alpha in alphas:

    # Entrenar exclusivamente con entrenamiento
    modelo = Ridge(alpha=alpha)

    modelo.fit(
        X_train_procesado,
        y_train
    )

    # Predicciones sobre validación
    predicciones = modelo.predict(
        X_val_procesado
    )

    # Calcular métricas
    mae = mean_absolute_error(
        y_val,
        predicciones
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            predicciones
        )
    )

    r2 = r2_score(
        y_val,
        predicciones
    )

    resultados_ridge.append({
        "alpha": alpha,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    modelos_ridge[alpha] = modelo

# Crear tabla de comparación
tabla_ridge = pd.DataFrame(
    resultados_ridge
).sort_values("RMSE").reset_index(drop=True)

display(
    tabla_ridge.style.format({
        "alpha": "{:.2f}",
        "MAE": "{:.2f}",
        "RMSE": "{:.2f}",
        "R2": "{:.4f}"
    })
)

# Seleccionar mejor alpha usando validación
mejor_alpha = float(
    tabla_ridge.iloc[0]["alpha"]
)

mejor_ridge = modelos_ridge[
    mejor_alpha
]

print("Mejor alpha:", mejor_alpha)

print(
    "RMSE de validación:",
    round(tabla_ridge.iloc[0]["RMSE"], 2)
)

# Verificaciones automáticas
assert len(tabla_ridge) == len(alphas)
assert np.isfinite(
    tabla_ridge[["MAE", "RMSE", "R2"]].to_numpy()
).all()

assert mejor_alpha in modelos_ridge

print("Optimización Ridge completada.")



### Interpretación de la optimización Ridge

Se evaluaron seis valores del parámetro de regularización `alpha`: 0.01, 0.1, 1, 10, 100 y 1000.

El mejor resultado en validación se obtuvo con `alpha = 0.1`, alcanzando un MAE de 730.47, un RMSE de 919.60 y un R² de -0.1677.

Los resultados evidencian que los valores bajos de regularización ofrecen un comportamiento similar, mientras que valores elevados incrementan considerablemente los errores de predicción.

El coeficiente R² negativo indica una capacidad predictiva limitada en el período de validación. Esto puede estar relacionado con cambios temporales en los patrones de demanda o con relaciones no lineales que Ridge no representa adecuadamente.

Por esta razón, se continuará con Random Forest Regressor para evaluar si un modelo basado en árboles mejora la capacidad predictiva.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 8
# Entrenamiento y optimización Random Forest Regressor
# ==========================================================

from sklearn.ensemble import RandomForestRegressor

resultados_rf = []
modelos_rf = {}

# Hiperparámetros a evaluar
n_estimators_opciones = [100, 200]
max_depth_opciones = [5, 10, None]
min_samples_leaf_opciones = [1, 3]

for n_arboles in n_estimators_opciones:
    for profundidad in max_depth_opciones:
        for min_hoja in min_samples_leaf_opciones:

            modelo = RandomForestRegressor(
                n_estimators=n_arboles,
                max_depth=profundidad,
                min_samples_leaf=min_hoja,
                random_state=SEMILLA,
                n_jobs=-1
            )

            # Entrenamiento
            modelo.fit(
                X_train_procesado,
                y_train
            )

            # Validación
            predicciones = modelo.predict(
                X_val_procesado
            )

            mae = mean_absolute_error(
                y_val,
                predicciones
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_val,
                    predicciones
                )
            )

            r2 = r2_score(
                y_val,
                predicciones
            )

            configuracion = (
                n_arboles,
                profundidad,
                min_hoja
            )

            resultados_rf.append({
                "n_estimators": n_arboles,
                "max_depth": str(profundidad),
                "min_samples_leaf": min_hoja,
                "MAE": mae,
                "RMSE": rmse,
                "R2": r2,
                "configuracion": configuracion
            })

            modelos_rf[configuracion] = modelo

# Tabla ordenada por menor RMSE
tabla_rf = pd.DataFrame(
    resultados_rf
).sort_values(
    "RMSE"
).reset_index(drop=True)

display(
    tabla_rf.drop(
        columns="configuracion"
    ).style.format({
        "MAE": "{:.2f}",
        "RMSE": "{:.2f}",
        "R2": "{:.4f}"
    })
)

# Seleccionar mejor modelo
mejor_config_rf = tabla_rf.iloc[0]["configuracion"]

mejor_rf = modelos_rf[
    mejor_config_rf
]

print("\nMejor configuración:", mejor_config_rf)

print(
    "RMSE de validación:",
    round(tabla_rf.iloc[0]["RMSE"], 2)
)

# Verificaciones
assert len(tabla_rf) == 12
assert np.isfinite(
    tabla_rf[["MAE", "RMSE", "R2"]].to_numpy()
).all()

print("Optimización Random Forest completada.")


In [ ]:

# ==========================================================
# FASE 3 - CELDA 9
# Comparación Ridge vs Random Forest
# ==========================================================

# Recuperar las métricas de los mejores modelos
comparacion_regresion = pd.DataFrame([
    {
        "Modelo": "Ridge",
        "MAE": tabla_ridge.iloc[0]["MAE"],
        "RMSE": tabla_ridge.iloc[0]["RMSE"],
        "R2": tabla_ridge.iloc[0]["R2"]
    },
    {
        "Modelo": "Random Forest",
        "MAE": tabla_rf.iloc[0]["MAE"],
        "RMSE": tabla_rf.iloc[0]["RMSE"],
        "R2": tabla_rf.iloc[0]["R2"]
    }
])

display(comparacion_regresion.round(4))

# Gráficos comparativos
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metrica in zip(
    axes,
    ["MAE", "RMSE", "R2"]
):
    sns.barplot(
        data=comparacion_regresion,
        x="Modelo",
        y=metrica,
        ax=ax
    )

    ax.set_title(f"Comparación de {metrica}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=15)

plt.suptitle(
    "Fase 3 - Comparación de regresión avanzada"
)

plt.tight_layout()

ruta_grafico = (
    RESULTADOS / "comparacion_regresion.png"
)

plt.savefig(
    ruta_grafico,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Selección del mejor modelo por RMSE
mejor_nombre = comparacion_regresion.loc[
    comparacion_regresion["RMSE"].idxmin(),
    "Modelo"
]

mejor_modelo_regresion = (
    mejor_rf
    if mejor_nombre == "Random Forest"
    else mejor_ridge
)

print("Mejor modelo:", mejor_nombre)
print("Gráfico guardado en:", ruta_grafico)

assert mejor_nombre == "Random Forest"
print("Comparación completada.")



### Comparación de los modelos de regresión avanzada

Se compararon Ridge y Random Forest Regressor utilizando las mismas particiones cronológicas y métricas de validación.

Ridge obtuvo un MAE de 730.47, RMSE de 919.60 y R² de -0.1677.

Random Forest obtuvo un MAE de 672.49, RMSE de 831.34 y R² de 0.0457, utilizando 200 árboles, profundidad máxima de 10 y mínimo de 3 muestras por hoja.

Random Forest presentó el mejor desempeño, reduciendo el RMSE aproximadamente un 9.6 % respecto a Ridge.

No obstante, el R² cercano a cero evidencia una capacidad explicativa limitada en validación. Se recomienda interpretar los resultados con cautela y considerar posibles cambios temporales en los patrones de demanda.

Random Forest se selecciona como el mejor modelo de regresión avanzada entre los candidatos evaluados.



## 6. Clasificación de la demanda diaria

Se transforma el problema de predicción continua en un problema de clasificación binaria:

- Clase 0: demanda baja.
- Clase 1: demanda alta.

El umbral se determina mediante la mediana de `cnt` calculada exclusivamente con el conjunto de entrenamiento.

Este criterio permite construir una variable objetivo reproducible sin utilizar información de validación o prueba para definir las clases.

Se entrenarán y compararán dos algoritmos:

1. K-Nearest Neighbors (KNN).
2. Random Forest Classifier.

La evaluación utilizará Accuracy, Precision, Recall, F1-Score y AUC-ROC.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 10
# Crear variable objetivo para clasificación
# ==========================================================

# Calcular el umbral SOLO con entrenamiento
umbral_demanda = float(y_train.median())

print(
    "Umbral de demanda:",
    round(umbral_demanda, 2),
    "alquileres diarios"
)

# Crear etiquetas binarias
# 0 = demanda baja; 1 = demanda alta
y_train_clf = (
    y_train >= umbral_demanda
).astype(int)

y_val_clf = (
    y_val >= umbral_demanda
).astype(int)

y_test_clf = (
    y_test >= umbral_demanda
).astype(int)

# Mostrar distribución de clases
distribucion_clases = pd.DataFrame({
    "Entrenamiento": y_train_clf.value_counts(),
    "Validación": y_val_clf.value_counts(),
    "Prueba": y_test_clf.value_counts()
}).fillna(0).astype(int)

distribucion_clases.index = (
    distribucion_clases.index.map({
        0: "Demanda baja",
        1: "Demanda alta"
    })
)

display(distribucion_clases)

# Verificaciones automáticas
assert set(y_train_clf.unique()) == {0, 1}
assert len(y_train_clf) == 511
assert len(y_val_clf) == 109
assert len(y_test_clf) == 111

assert (
    y_train_clf.index.equals(X_train.index)
    and y_val_clf.index.equals(X_val.index)
    and y_test_clf.index.equals(X_test.index)
)

print("Variable objetivo de clasificación creada.")


In [ ]:

# ==========================================================
# FASE 3 - CELDA 11
# Diagnóstico de distribución de clases
# ==========================================================

from sklearn.metrics import roc_auc_score

conjuntos_clf = {
    "Entrenamiento": y_train_clf,
    "Validación": y_val_clf,
    "Prueba": y_test_clf
}

for nombre, etiquetas in conjuntos_clf.items():

    conteos = etiquetas.value_counts().sort_index()
    clases = etiquetas.nunique()

    print(f"\n{nombre}")
    print("Clases presentes:", clases)
    print("Distribución:", conteos.to_dict())

    if clases < 2:
        print(
            "ADVERTENCIA: AUC-ROC no está definida "
            "porque solo existe una clase real."
        )
    else:
        print(
            "AUC-ROC podrá calcularse cuando "
            "dispongamos de probabilidades del modelo."
        )

# Verificar el desequilibrio detectado
assert y_train_clf.nunique() == 2
assert y_val_clf.nunique() == 1
assert y_test_clf.nunique() == 2

print("\nDiagnóstico de clasificación completado.")



### Análisis del desbalance y cambio temporal de clases

El umbral de clasificación se estableció en 3.974 alquileres diarios, utilizando exclusivamente la mediana del conjunto de entrenamiento.

La distribución de clases en entrenamiento fue equilibrada: 255 días de demanda baja y 256 de demanda alta.

Sin embargo, el conjunto de validación presentó únicamente días de demanda alta (109 registros), mientras que el conjunto de prueba incluyó 21 días de demanda baja y 90 de demanda alta.

Este comportamiento evidencia un cambio temporal en la distribución de la variable objetivo y limita la interpretación de algunas métricas de clasificación.

En particular, AUC-ROC no está definida para validación debido a la ausencia de una de las clases.

Para seleccionar hiperparámetros se utilizará validación temporal interna sobre entrenamiento. El conjunto de prueba permanecerá reservado para la evaluación final.



## 7. Validación temporal para clasificación

Debido a que la validación externa contiene únicamente observaciones de demanda alta, se implementa una validación temporal interna sobre el conjunto de entrenamiento.

Se utiliza `TimeSeriesSplit` para respetar el orden cronológico. En cada partición, los datos utilizados para entrenar son anteriores a los datos utilizados para validar.

Se verificará la presencia de ambas clases en cada partición. AUC-ROC solo se calculará cuando existan las dos clases reales.

El escalamiento y la codificación categórica se ajustarán nuevamente dentro de cada partición mediante un `Pipeline`, evitando fuga de información durante la optimización.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 12
# Diagnóstico de validación temporal interna
# ==========================================================

from sklearn.model_selection import TimeSeriesSplit

# Cinco particiones temporales progresivas
tscv_clf = TimeSeriesSplit(n_splits=5)

resultados_folds = []

for numero, (idx_train, idx_val) in enumerate(
    tscv_clf.split(X_train),
    start=1
):
    y_fold_train = y_train_clf.iloc[idx_train]
    y_fold_val = y_train_clf.iloc[idx_val]

    clases_train = y_fold_train.nunique()
    clases_val = y_fold_val.nunique()

    resultados_folds.append({
        "Fold": numero,
        "N_entrenamiento": len(idx_train),
        "N_validacion": len(idx_val),
        "Clases_entrenamiento": clases_train,
        "Clases_validacion": clases_val,
        "AUC_calculable": clases_val == 2,
        "Fold_utilizable": (
            clases_train == 2 and clases_val == 2
        )
    })

tabla_folds = pd.DataFrame(resultados_folds)

display(tabla_folds)

folds_validos = tabla_folds[
    tabla_folds["Fold_utilizable"]
]

print(
    "Particiones utilizables:",
    len(folds_validos),
    "de",
    len(tabla_folds)
)

assert len(tabla_folds) == 5

print("Diagnóstico temporal completado.")



### 7.1. Clasificación mediante K-Nearest Neighbors (KNN)

Se implementa KNN como primer algoritmo de clasificación binaria de la demanda de bicicletas.

Se evalúan diferentes cantidades de vecinos (`k = 3, 5, 7, 9, 11, 15`) mediante validación temporal interna.

Se excluye el primer fold porque contiene una sola clase en entrenamiento, lo que impide una evaluación comparable del problema binario.

En cada fold se ajusta nuevamente el preprocesamiento sobre sus datos de entrenamiento, evitando fuga de información.

El criterio principal de selección será el F1-Score promedio, complementado con Accuracy, Precision, Recall y AUC-ROC.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 13
# Optimización de KNN con validación temporal
# ==========================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Conservar únicamente folds con ambas clases
folds_clf_validos = []

for idx_train, idx_val in tscv_clf.split(X_train):

    etiquetas_train = y_train_clf.iloc[idx_train]
    etiquetas_val = y_train_clf.iloc[idx_val]

    if (
        etiquetas_train.nunique() == 2
        and etiquetas_val.nunique() == 2
    ):
        folds_clf_validos.append(
            (idx_train, idx_val)
        )

assert len(folds_clf_validos) == 4

# Hiperparámetros a comparar
valores_k = [3, 5, 7, 9, 11, 15]

resultados_knn = []

for k in valores_k:

    metricas_folds = []

    for idx_train, idx_val in folds_clf_validos:

        X_fold_train = X_train.iloc[idx_train]
        X_fold_val = X_train.iloc[idx_val]

        y_fold_train = y_train_clf.iloc[idx_train]
        y_fold_val = y_train_clf.iloc[idx_val]

        # Pipeline: preprocesamiento ajustado
        # exclusivamente en cada fold de entrenamiento
        pipeline_knn = Pipeline([
            (
                "preprocesamiento",
                clone(preprocesador)
            ),
            (
                "clasificador",
                KNeighborsClassifier(
                    n_neighbors=k
                )
            )
        ])

        pipeline_knn.fit(
            X_fold_train,
            y_fold_train
        )

        pred = pipeline_knn.predict(
            X_fold_val
        )

        prob = pipeline_knn.predict_proba(
            X_fold_val
        )[:, 1]

        metricas_folds.append({
            "Accuracy": accuracy_score(
                y_fold_val, pred
            ),
            "Precision": precision_score(
                y_fold_val, pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_fold_val, pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_fold_val, pred,
                zero_division=0
            ),
            "AUC_ROC": roc_auc_score(
                y_fold_val, prob
            )
        })

    promedios = pd.DataFrame(
        metricas_folds
    ).mean()

    resultados_knn.append({
        "k": k,
        "Accuracy": promedios["Accuracy"],
        "Precision": promedios["Precision"],
        "Recall": promedios["Recall"],
        "F1": promedios["F1"],
        "AUC_ROC": promedios["AUC_ROC"]
    })

# Comparación de configuraciones
tabla_knn = pd.DataFrame(
    resultados_knn
).sort_values(
    ["F1", "AUC_ROC"],
    ascending=False
).reset_index(drop=True)

display(
    tabla_knn.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "AUC_ROC": "{:.4f}"
    })
)

# Mejor configuración
mejor_k = int(tabla_knn.iloc[0]["k"])

print("Mejor k:", mejor_k)
print(
    "F1 promedio:",
    round(tabla_knn.iloc[0]["F1"], 4)
)

assert len(tabla_knn) == len(valores_k)
assert np.isfinite(
    tabla_knn[
        ["Accuracy", "Precision", "Recall", "F1", "AUC_ROC"]
    ].to_numpy()
).all()

print("Optimización KNN completada.")



### Interpretación de los resultados KNN

Se evaluaron seis configuraciones de KNN utilizando validación temporal interna con cuatro particiones válidas.

El mejor modelo se obtuvo con `k = 11`, alcanzando un Accuracy promedio de 0.7471, Precision de 0.7088, Recall de 0.6224, F1-Score de 0.6428 y AUC-ROC de 0.6756.

Aunque algunas configuraciones alcanzaron mayor Accuracy, se seleccionó `k = 11` por presentar el mayor F1-Score promedio.

Los resultados sugieren una capacidad moderada para distinguir días de demanda alta y baja. La evaluación final sobre datos futuros permitirá determinar si este comportamiento se mantiene fuera del conjunto de entrenamiento.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 14
# Optimización de Random Forest Classifier
# ==========================================================

from sklearn.ensemble import RandomForestClassifier

resultados_rf_clf = []

# Configuraciones a evaluar
n_estimators_clf = [100, 200]
max_depth_clf = [5, 10, None]
min_samples_leaf_clf = [1, 3]

for n_arboles in n_estimators_clf:
    for profundidad in max_depth_clf:
        for min_hoja in min_samples_leaf_clf:

            metricas_folds = []

            for idx_train, idx_val in folds_clf_validos:

                X_fold_train = X_train.iloc[idx_train]
                X_fold_val = X_train.iloc[idx_val]

                y_fold_train = y_train_clf.iloc[idx_train]
                y_fold_val = y_train_clf.iloc[idx_val]

                # Pipeline ajustado dentro de cada fold
                pipeline_rf = Pipeline([
                    (
                        "preprocesamiento",
                        clone(preprocesador)
                    ),
                    (
                        "clasificador",
                        RandomForestClassifier(
                            n_estimators=n_arboles,
                            max_depth=profundidad,
                            min_samples_leaf=min_hoja,
                            random_state=SEMILLA,
                            n_jobs=-1
                        )
                    )
                ])

                pipeline_rf.fit(
                    X_fold_train,
                    y_fold_train
                )

                pred = pipeline_rf.predict(
                    X_fold_val
                )

                prob = pipeline_rf.predict_proba(
                    X_fold_val
                )[:, 1]

                metricas_folds.append({
                    "Accuracy": accuracy_score(
                        y_fold_val, pred
                    ),
                    "Precision": precision_score(
                        y_fold_val, pred,
                        zero_division=0
                    ),
                    "Recall": recall_score(
                        y_fold_val, pred,
                        zero_division=0
                    ),
                    "F1": f1_score(
                        y_fold_val, pred,
                        zero_division=0
                    ),
                    "AUC_ROC": roc_auc_score(
                        y_fold_val, prob
                    )
                })

            promedios = pd.DataFrame(
                metricas_folds
            ).mean()

            resultados_rf_clf.append({
                "n_estimators": n_arboles,
                "max_depth": str(profundidad),
                "min_samples_leaf": min_hoja,
                "Accuracy": promedios["Accuracy"],
                "Precision": promedios["Precision"],
                "Recall": promedios["Recall"],
                "F1": promedios["F1"],
                "AUC_ROC": promedios["AUC_ROC"]
            })

# Tabla comparativa
tabla_rf_clf = pd.DataFrame(
    resultados_rf_clf
).sort_values(
    ["F1", "AUC_ROC"],
    ascending=False
).reset_index(drop=True)

display(
    tabla_rf_clf.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "AUC_ROC": "{:.4f}"
    })
)

# Seleccionar mejor configuración
mejor_config_rf_clf = tabla_rf_clf.iloc[0]

print(
    "Mejor configuración:",
    int(mejor_config_rf_clf["n_estimators"]),
    mejor_config_rf_clf["max_depth"],
    int(mejor_config_rf_clf["min_samples_leaf"])
)

print(
    "F1 promedio:",
    round(mejor_config_rf_clf["F1"], 4)
)

# Verificaciones automáticas
assert len(tabla_rf_clf) == 12

assert np.isfinite(
    tabla_rf_clf[
        ["Accuracy", "Precision", "Recall", "F1", "AUC_ROC"]
    ].to_numpy()
).all()

print("Optimización Random Forest Classifier completada.")


In [ ]:

# ==========================================================
# FASE 3 - CELDA 15
# Comparación KNN vs Random Forest Classifier
# ==========================================================

# Recuperar mejores configuraciones
mejor_knn_fila = tabla_knn.iloc[0]
mejor_rf_fila = tabla_rf_clf.iloc[0]

metricas_clf = [
    "Accuracy", "Precision", "Recall", "F1", "AUC_ROC"
]

comparacion_clasificacion = pd.DataFrame([
    {
        "Modelo": "KNN",
        **{
            metrica: float(mejor_knn_fila[metrica])
            for metrica in metricas_clf
        }
    },
    {
        "Modelo": "Random Forest",
        **{
            metrica: float(mejor_rf_fila[metrica])
            for metrica in metricas_clf
        }
    }
])

display(
    comparacion_clasificacion.style.format(
        {m: "{:.4f}" for m in metricas_clf}
    )
)

# Gráfico comparativo
datos_grafico = comparacion_clasificacion.melt(
    id_vars="Modelo",
    value_vars=metricas_clf,
    var_name="Métrica",
    value_name="Valor"
)

plt.figure(figsize=(12, 5))

sns.barplot(
    data=datos_grafico,
    x="Métrica",
    y="Valor",
    hue="Modelo"
)

plt.title(
    "Comparación de modelos de clasificación"
)
plt.ylabel("Puntuación promedio")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

ruta_grafico_clf = (
    RESULTADOS / "comparacion_clasificacion.png"
)

plt.savefig(
    ruta_grafico_clf,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Selección por F1-Score
indice_mejor = (
    comparacion_clasificacion["F1"].idxmax()
)

mejor_nombre_clf = (
    comparacion_clasificacion.loc[
        indice_mejor, "Modelo"
    ]
)

print("Mejor clasificador:", mejor_nombre_clf)
print("Gráfico guardado en:", ruta_grafico_clf)

assert mejor_nombre_clf == "KNN"

print("Comparación de clasificación completada.")



### Comparación de los algoritmos de clasificación

Se compararon K-Nearest Neighbors y Random Forest Classifier mediante cuatro particiones de validación temporal interna.

KNN, con 11 vecinos, obtuvo el mayor F1-Score promedio (0.6428), además de mejores valores de Precision (0.7088) y Recall (0.6224).

Random Forest Classifier, configurado con 200 árboles, profundidad máxima de 5 y mínimo de una muestra por hoja, alcanzó mayor Accuracy (0.7824) y AUC-ROC (0.7523).

Debido a que el criterio principal de selección fue F1-Score, se seleccionó KNN como mejor clasificador en la validación temporal interna.

No obstante, Random Forest presentó una mayor capacidad de discriminación según AUC-ROC. Esta diferencia evidencia la importancia de utilizar múltiples métricas al evaluar algoritmos de clasificación.

La selección es provisional hasta realizar la evaluación final sobre el conjunto de prueba reservado.



## 8. Aprendizaje no supervisado: segmentación de días

Se aplicarán dos algoritmos de agrupamiento:

1. K-Means.
2. Clustering jerárquico aglomerativo (Agglomerative Clustering).

El objetivo es identificar patrones de días con características meteorológicas y de calendario similares.

Para construir los clústeres se utilizarán variables independientes, excluyendo la cantidad total de alquileres (`cnt`) y sus componentes (`casual` y `registered`).

Se evaluará el número de clústeres mediante el método del codo y el coeficiente de silueta.

Posteriormente, se interpretarán los grupos utilizando estadísticas descriptivas de la demanda de bicicletas.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 16
# Preparación de datos para aprendizaje no supervisado
# ==========================================================

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Usar el conjunto completo para análisis descriptivo
# no supervisado. No se entrenan modelos predictivos aquí.
df_cluster = crear_caracteristicas(
    df_modelo.copy()
)

# Variables meteorológicas
numericas_cluster = [
    "temp",
    "hum",
    "windspeed"
]

# Variables de calendario y clima
categoricas_cluster = [
    "season",
    "weathersit",
    "workingday",
    "yr"
]

variables_cluster = (
    numericas_cluster + categoricas_cluster
)

X_cluster = df_cluster[
    variables_cluster
].copy()

# Transformación de variables
preprocesador_cluster = ColumnTransformer([
    (
        "numericas",
        StandardScaler(),
        numericas_cluster
    ),
    (
        "categoricas",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ),
        categoricas_cluster
    )
])

X_cluster_procesado = (
    preprocesador_cluster.fit_transform(
        X_cluster
    )
)

print(
    "Registros para clustering:",
    X_cluster_procesado.shape[0]
)

print(
    "Características transformadas:",
    X_cluster_procesado.shape[1]
)

print(
    "Variables originales:",
    variables_cluster
)

# Validaciones
assert X_cluster_procesado.shape[0] == 731

assert np.isfinite(
    X_cluster_procesado
).all()

assert not {
    "cnt", "casual", "registered"
}.intersection(variables_cluster)

print(
    "Preparación para clustering completada."
)



### 8.1. Selección del número de clústeres mediante K-Means

Se evaluarán configuraciones de K-Means con valores de K entre 2 y 10.

Para cada configuración se calcularán dos indicadores:

- **Inercia:** suma de las distancias cuadráticas de las observaciones a los centroides de sus respectivos clústeres.
- **Coeficiente de silueta:** indicador de cohesión y separación de los grupos.

El método del codo permitirá analizar la disminución de la inercia al aumentar K, mientras que el coeficiente de silueta facilitará la selección cuantitativa de una configuración.

Se utilizará una semilla fija para garantizar reproducibilidad.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 17
# K-Means: método del codo y coeficiente de silueta
# ==========================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

valores_k_cluster = range(2, 11)

resultados_kmeans = []

for k in valores_k_cluster:

    modelo = KMeans(
        n_clusters=k,
        random_state=SEMILLA,
        n_init=10
    )

    etiquetas = modelo.fit_predict(
        X_cluster_procesado
    )

    inercia = modelo.inertia_

    silueta = silhouette_score(
        X_cluster_procesado,
        etiquetas
    )

    resultados_kmeans.append({
        "K": k,
        "Inercia": inercia,
        "Silueta": silueta
    })

tabla_kmeans = pd.DataFrame(
    resultados_kmeans
)

display(
    tabla_kmeans.style.format({
        "Inercia": "{:.2f}",
        "Silueta": "{:.4f}"
    })
)

# Selección automática por silueta
mejor_k_cluster = int(
    tabla_kmeans.loc[
        tabla_kmeans["Silueta"].idxmax(),
        "K"
    ]
)

print(
    "Mejor K según silueta:",
    mejor_k_cluster
)

# Visualización
fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4)
)

# Método del codo
axes[0].plot(
    tabla_kmeans["K"],
    tabla_kmeans["Inercia"],
    marker="o"
)

axes[0].set_title(
    "Método del codo - K-Means"
)
axes[0].set_xlabel(
    "Número de clústeres (K)"
)
axes[0].set_ylabel("Inercia")
axes[0].grid(alpha=0.3)

# Coeficiente de silueta
axes[1].plot(
    tabla_kmeans["K"],
    tabla_kmeans["Silueta"],
    marker="o"
)

axes[1].axvline(
    mejor_k_cluster,
    color="red",
    linestyle="--",
    label=f"Mejor K = {mejor_k_cluster}"
)

axes[1].set_title(
    "Coeficiente de silueta - K-Means"
)
axes[1].set_xlabel(
    "Número de clústeres (K)"
)
axes[1].set_ylabel(
    "Silueta promedio"
)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()

ruta_kmeans = (
    RESULTADOS / "kmeans_codo_silueta.png"
)

plt.savefig(
    ruta_kmeans,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Verificaciones
assert len(tabla_kmeans) == 9
assert 2 <= mejor_k_cluster <= 10

assert np.isfinite(
    tabla_kmeans[
        ["Inercia", "Silueta"]
    ].to_numpy()
).all()

print(
    "Análisis K-Means completado."
)
print(
    "Gráfico guardado en:",
    ruta_kmeans
)



### Interpretación del método del codo y silueta

Se evaluaron nueve configuraciones de K-Means, desde K = 2 hasta K = 10.

El mayor coeficiente de silueta fue 0.1918, correspondiente a K = 2. La configuración K = 3 obtuvo un valor cercano de 0.1902.

La curva del codo mostró una disminución progresiva de la inercia, sin un punto de inflexión completamente definido.

Se seleccionó K = 2 por presentar el mayor coeficiente de silueta.

Sin embargo, el valor relativamente bajo de este indicador sugiere una separación limitada entre los grupos. Por tanto, los clústeres se interpretarán como segmentos exploratorios y no como categorías claramente diferenciadas.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 18
# K-Means definitivo y perfil de los clústeres
# ==========================================================

from sklearn.cluster import KMeans

# Entrenar con el K seleccionado
modelo_kmeans_final = KMeans(
    n_clusters=mejor_k_cluster,
    random_state=SEMILLA,
    n_init=10
)

etiquetas_kmeans = modelo_kmeans_final.fit_predict(
    X_cluster_procesado
)

# Incorporar etiquetas al dataframe descriptivo
df_cluster["cluster_kmeans"] = etiquetas_kmeans

# Tamaño de cada clúster
tamano_clusters = (
    df_cluster["cluster_kmeans"]
    .value_counts()
    .sort_index()
)

print("Distribución de observaciones:")
display(tamano_clusters.to_frame("Cantidad de días"))

# Perfil de los clústeres
perfil_kmeans = (
    df_cluster
    .groupby("cluster_kmeans")
    .agg(
        dias=("cnt", "size"),
        demanda_promedio=("cnt", "mean"),
        demanda_mediana=("cnt", "median"),
        temperatura_promedio=("temp", "mean"),
        humedad_promedio=("hum", "mean"),
        viento_promedio=("windspeed", "mean"),
        proporcion_laborales=("workingday", "mean")
    )
    .round(3)
)

print("\nPerfil de los clústeres:")
display(perfil_kmeans)

# Gráfico de demanda promedio por clúster
plt.figure(figsize=(8, 4))

sns.barplot(
    data=df_cluster,
    x="cluster_kmeans",
    y="cnt",
    estimator="mean",
    errorbar=None
)

plt.title(
    "Demanda promedio de bicicletas por clúster"
)
plt.xlabel("Clúster K-Means")
plt.ylabel("Alquileres diarios promedio")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

ruta_perfil_kmeans = (
    RESULTADOS / "perfil_kmeans.png"
)

plt.savefig(
    ruta_perfil_kmeans,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Verificaciones
assert len(etiquetas_kmeans) == 731

assert (
    df_cluster["cluster_kmeans"].nunique()
    == mejor_k_cluster
)

assert tamano_clusters.sum() == 731

print("Caracterización K-Means completada.")



### Interpretación de los clústeres obtenidos con K-Means

El algoritmo K-Means identificó dos grupos de días a partir de características meteorológicas y de calendario.

El clúster 0 agrupó 293 observaciones y presentó una demanda promedio de 3540.51 alquileres diarios. Se caracterizó por temperaturas más bajas y mayor velocidad del viento.

El clúster 1 agrupó 438 observaciones y presentó una demanda promedio de 5149.11 alquileres diarios. Se caracterizó por temperaturas más elevadas, mayor humedad y menor velocidad del viento.

La diferencia entre ambos grupos fue de aproximadamente 1609 alquileres diarios, equivalente a un incremento del 45.4 % respecto al clúster 0.

Estos resultados sugieren una asociación entre las condiciones meteorológicas y la demanda del sistema de bicicletas.

No obstante, el coeficiente de silueta de 0.1918 indica una separación limitada entre los clústeres. Por ello, la segmentación debe interpretarse como exploratoria y no como una clasificación definitiva de los días.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 19
# Clustering jerárquico aglomerativo
# ==========================================================

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score
)

# Modelo jerárquico con el mismo número de clústeres
modelo_aglomerativo = AgglomerativeClustering(
    n_clusters=mejor_k_cluster,
    linkage="ward"
)

etiquetas_aglomerativo = (
    modelo_aglomerativo.fit_predict(
        X_cluster_procesado
    )
)

# Incorporar etiquetas al dataframe
df_cluster["cluster_aglomerativo"] = (
    etiquetas_aglomerativo
)

# Silueta de ambos algoritmos
silueta_kmeans = silhouette_score(
    X_cluster_procesado,
    etiquetas_kmeans
)

silueta_aglomerativo = silhouette_score(
    X_cluster_procesado,
    etiquetas_aglomerativo
)

# Coincidencia de agrupamientos, independiente
# de los números asignados a las etiquetas
ari = adjusted_rand_score(
    etiquetas_kmeans,
    etiquetas_aglomerativo
)

comparacion_clustering = pd.DataFrame([
    {
        "Algoritmo": "K-Means",
        "Clusters": mejor_k_cluster,
        "Silueta": silueta_kmeans
    },
    {
        "Algoritmo": "Agglomerative",
        "Clusters": mejor_k_cluster,
        "Silueta": silueta_aglomerativo
    }
])

print("Comparación de algoritmos:")
display(
    comparacion_clustering.style.format({
        "Silueta": "{:.4f}"
    })
)

print(
    "Índice Rand ajustado (ARI):",
    round(ari, 4)
)

print("\nDistribución Agglomerative:")
display(
    df_cluster["cluster_aglomerativo"]
    .value_counts()
    .sort_index()
    .to_frame("Cantidad de días")
)

# Verificaciones
assert len(etiquetas_aglomerativo) == 731
assert len(np.unique(etiquetas_aglomerativo)) == 2

assert np.isfinite(
    [silueta_kmeans, silueta_aglomerativo, ari]
).all()

print("Clustering jerárquico completado.")



### Comparación de algoritmos de aprendizaje no supervisado

Se compararon K-Means y Agglomerative Clustering utilizando dos clústeres y las mismas 14 características transformadas.

K-Means obtuvo un coeficiente de silueta de 0.1918, mientras que Agglomerative Clustering alcanzó 0.1543.

Por tanto, K-Means presentó una mejor separación relativa de los grupos según el coeficiente de silueta.

El índice Rand ajustado (ARI) fue de 0.4276, lo cual indica una concordancia parcial entre las asignaciones generadas por ambos algoritmos.

Aunque K-Means obtuvo el mejor resultado, los valores de silueta fueron bajos en ambos casos. Esto sugiere que los datos no presentan una estructura de agrupamiento claramente definida con las características y métodos utilizados.

Los resultados pueden utilizarse para explorar perfiles meteorológicos y de demanda, evitando interpretar los clústeres como categorías definitivas.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 20
# Perfil de clústeres de Agglomerative Clustering
# ==========================================================

# Construir perfil descriptivo
perfil_aglomerativo = (
    df_cluster
    .groupby("cluster_aglomerativo")
    .agg(
        dias=("cnt", "size"),
        demanda_promedio=("cnt", "mean"),
        demanda_mediana=("cnt", "median"),
        temperatura_promedio=("temp", "mean"),
        humedad_promedio=("hum", "mean"),
        viento_promedio=("windspeed", "mean"),
        proporcion_laborales=("workingday", "mean")
    )
    .round(3)
)

print("Perfil de Agglomerative Clustering:")
display(perfil_aglomerativo)

# Comparación gráfica de la demanda promedio
fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 4)
)

sns.barplot(
    data=df_cluster,
    x="cluster_kmeans",
    y="cnt",
    estimator="mean",
    errorbar=None,
    ax=axes[0]
)

axes[0].set_title("K-Means")
axes[0].set_xlabel("Clúster")
axes[0].set_ylabel("Alquileres diarios promedio")

sns.barplot(
    data=df_cluster,
    x="cluster_aglomerativo",
    y="cnt",
    estimator="mean",
    errorbar=None,
    ax=axes[1]
)

axes[1].set_title("Agglomerative Clustering")
axes[1].set_xlabel("Clúster")
axes[1].set_ylabel("Alquileres diarios promedio")

for ax in axes:
    ax.grid(axis="y", alpha=0.25)

plt.suptitle(
    "Comparación de perfiles de demanda por algoritmo"
)

plt.tight_layout()

ruta_comparacion_clusters = (
    RESULTADOS / "comparacion_perfiles_clusters.png"
)

plt.savefig(
    ruta_comparacion_clusters,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Verificaciones
assert perfil_aglomerativo["dias"].sum() == 731
assert len(perfil_aglomerativo) == 2

print("Caracterización de Agglomerative completada.")



### Interpretación de Agglomerative Clustering

El algoritmo Agglomerative Clustering identificó dos grupos de días.

El clúster 0 agrupó 446 observaciones, con una demanda promedio de 4830.42 alquileres diarios. Se caracterizó por temperaturas y niveles de humedad superiores, así como menor velocidad del viento.

El clúster 1 agrupó 285 observaciones, con una demanda promedio de 3994.08 alquileres diarios. Presentó temperaturas más bajas y mayor velocidad del viento.

La diferencia entre ambos grupos fue de aproximadamente 836 alquileres diarios.

Los resultados muestran patrones descriptivos similares a los encontrados mediante K-Means, aunque con diferentes asignaciones de observaciones.

K-Means obtuvo un coeficiente de silueta superior (0.1918 frente a 0.1543), por lo que se seleccionó como el algoritmo de agrupamiento preferido según este criterio.

No obstante, ambos algoritmos presentaron una separación limitada de los grupos, por lo que los resultados deben interpretarse como exploratorios.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 21
# Interpretabilidad: importancia por permutación
# ==========================================================

from sklearn.inspection import permutation_importance

# Mejor modelo de regresión ya seleccionado
assert mejor_nombre == "Random Forest"

# Importancia sobre validación, no sobre prueba
importancias = permutation_importance(
    mejor_rf,
    X_val_procesado,
    y_val,
    scoring="neg_root_mean_squared_error",
    n_repeats=10,
    random_state=SEMILLA,
    n_jobs=-1
)

# Nombres de las características transformadas
nombres_variables = (
    preprocesador.get_feature_names_out()
)

tabla_importancias = pd.DataFrame({
    "Variable": nombres_variables,
    "Importancia_RMSE": importancias.importances_mean,
    "Desviacion": importancias.importances_std
}).sort_values(
    "Importancia_RMSE",
    ascending=False
).reset_index(drop=True)

print("Importancia por permutación:")
display(tabla_importancias.head(12).round(3))

# Visualización de las 10 variables más importantes
top_importancias = (
    tabla_importancias.head(10)
    .sort_values("Importancia_RMSE")
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_importancias["Variable"],
    top_importancias["Importancia_RMSE"]
)

plt.axvline(0, color="black", linewidth=0.8)
plt.title(
    "Importancia por permutación - Random Forest"
)
plt.xlabel(
    "Incremento promedio del RMSE al permutar"
)
plt.ylabel("Característica transformada")
plt.tight_layout()

ruta_importancias = (
    RESULTADOS / "importancia_variables_regresion.png"
)

plt.savefig(
    ruta_importancias,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

assert len(tabla_importancias) == (
    X_val_procesado.shape[1]
)

assert np.isfinite(
    tabla_importancias["Importancia_RMSE"]
).all()

print("Análisis de interpretabilidad completado.")



### Interpretabilidad del modelo Random Forest Regressor

Se utilizó la técnica de importancia por permutación sobre el conjunto de validación para identificar las características con mayor contribución predictiva.

La humedad presentó la mayor importancia promedio, con un incremento del RMSE de 102.359 al permutar sus valores. Le siguió la velocidad del viento, con 40.503, y la interacción temperatura-humedad, con 8.681.

Las variables relacionadas con estaciones y condiciones meteorológicas presentaron contribuciones menores.

Estos resultados sugieren que la humedad y el viento fueron características relevantes para las predicciones del modelo en el período de validación.

No obstante, la importancia por permutación no permite establecer relaciones causales y puede verse afectada por la correlación entre variables y por las características particulares del conjunto evaluado.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 22
# Evaluación final de modelos de regresión
# ==========================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Modelos previamente seleccionados
modelos_regresion_final = {
    "Ridge": mejor_ridge,
    "Random Forest": mejor_rf
}

resultados_regresion_test = []
predicciones_regresion_test = {}

for nombre, modelo in modelos_regresion_final.items():

    # Evaluación sobre los 111 días reservados
    pred = modelo.predict(X_test_procesado)

    predicciones_regresion_test[nombre] = pred

    resultados_regresion_test.append({
        "Modelo": nombre,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, pred)
        ),
        "R2": r2_score(y_test, pred)
    })

tabla_regresion_test = pd.DataFrame(
    resultados_regresion_test
).sort_values(
    "RMSE"
).reset_index(drop=True)

print("Evaluación final de regresión:")
display(
    tabla_regresion_test.style.format({
        "MAE": "{:.2f}",
        "RMSE": "{:.2f}",
        "R2": "{:.4f}"
    })
)

# Comparación visual de predicciones
fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5)
)

for ax, (nombre, pred) in zip(
    axes,
    predicciones_regresion_test.items()
):

    ax.scatter(
        y_test,
        pred,
        alpha=0.65
    )

    minimo = min(y_test.min(), pred.min())
    maximo = max(y_test.max(), pred.max())

    ax.plot(
        [minimo, maximo],
        [minimo, maximo],
        linestyle="--",
        color="red"
    )

    ax.set_title(nombre)
    ax.set_xlabel("Demanda real")
    ax.set_ylabel("Demanda predicha")
    ax.grid(alpha=0.25)

plt.suptitle(
    "Evaluación final de regresión - Datos de prueba"
)

plt.tight_layout()

ruta_regresion_test = (
    RESULTADOS / "evaluacion_final_regresion.png"
)

plt.savefig(
    ruta_regresion_test,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Verificaciones
assert len(y_test) == 111

assert len(tabla_regresion_test) == 2

assert np.isfinite(
    tabla_regresion_test[
        ["MAE", "RMSE", "R2"]
    ].to_numpy()
).all()

print("Evaluación final de regresión completada.")



### Evaluación final de los modelos de regresión

Se evaluaron los modelos Ridge y Random Forest Regressor sobre 111 registros de prueba, que no fueron utilizados durante el entrenamiento ni la selección de hiperparámetros.

Ridge obtuvo un MAE de 938.57, RMSE de 1232.48 y R² de 0.6215.

Random Forest obtuvo un MAE de 1210.53, RMSE de 1386.53 y R² de 0.5210.

Aunque Random Forest había presentado mejores resultados durante la validación, Ridge alcanzó un mejor desempeño en el conjunto de prueba, con un RMSE aproximadamente 11.1 % inferior.

Este comportamiento evidencia la importancia de evaluar los modelos sobre datos no utilizados durante la optimización, especialmente cuando existen cambios temporales en la distribución de la demanda.

Los resultados también muestran que ambos modelos presentan errores relevantes en algunas observaciones, por lo que sería conveniente explorar nuevas características y estrategias de validación en trabajos futuros.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 23
# Evaluación final de clasificación
# ==========================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Reconstruir modelos con los parámetros seleccionados
modelos_clasificacion_final = {
    "KNN": Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("clasificador", KNeighborsClassifier(
            n_neighbors=mejor_k
        ))
    ]),
    "Random Forest": Pipeline([
        ("preprocesamiento", clone(preprocesador)),
        ("clasificador", RandomForestClassifier(
            n_estimators=int(
                mejor_config_rf_clf["n_estimators"]
            ),
            max_depth=(
                None
                if mejor_config_rf_clf["max_depth"] == "None"
                else int(mejor_config_rf_clf["max_depth"])
            ),
            min_samples_leaf=int(
                mejor_config_rf_clf["min_samples_leaf"]
            ),
            random_state=SEMILLA,
            n_jobs=-1
        ))
    ])
}

resultados_clasificacion_test = []
predicciones_clasificacion_test = {}

# Entrenar únicamente con el conjunto de entrenamiento
for nombre, modelo in modelos_clasificacion_final.items():

    modelo.fit(X_train, y_train_clf)

    pred = modelo.predict(X_test)
    prob = modelo.predict_proba(X_test)[:, 1]

    predicciones_clasificacion_test[nombre] = pred

    resultados_clasificacion_test.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test_clf, pred),
        "Precision": precision_score(
            y_test_clf, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test_clf, pred, zero_division=0
        ),
        "F1": f1_score(
            y_test_clf, pred, zero_division=0
        ),
        "AUC_ROC": roc_auc_score(
            y_test_clf, prob
        )
    })

tabla_clasificacion_test = pd.DataFrame(
    resultados_clasificacion_test
).sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

print("Evaluación final de clasificación:")
display(
    tabla_clasificacion_test.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
        "AUC_ROC": "{:.4f}"
    })
)

# Matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (nombre, pred) in zip(
    axes,
    predicciones_clasificacion_test.items()
):
    matriz = confusion_matrix(
        y_test_clf,
        pred,
        labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=["Baja", "Alta"]
    ).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False
    )

    ax.set_title(nombre)

plt.suptitle(
    "Matrices de confusión - Conjunto de prueba"
)
plt.tight_layout()

ruta_clasificacion_test = (
    RESULTADOS / "evaluacion_final_clasificacion.png"
)

plt.savefig(
    ruta_clasificacion_test,
    dpi=160,
    bbox_inches="tight"
)

plt.show()

# Verificaciones
assert len(y_test_clf) == 111
assert y_test_clf.nunique() == 2
assert len(tabla_clasificacion_test) == 2

assert np.isfinite(
    tabla_clasificacion_test[
        ["Accuracy", "Precision", "Recall", "F1", "AUC_ROC"]
    ].to_numpy()
).all()

print("Evaluación final de clasificación completada.")



### Evaluación final de los modelos de clasificación

Se evaluaron KNN y Random Forest Classifier sobre los 111 registros del conjunto de prueba, compuesto por 21 días de demanda baja y 90 días de demanda alta.

KNN obtuvo un Accuracy de 0.6306, Precision de 0.9623, Recall de 0.5667, F1-Score de 0.7133 y AUC-ROC de 0.8730.

Random Forest obtuvo un Accuracy de 0.7027, Precision de 0.9524, Recall de 0.6667, F1-Score de 0.7843 y AUC-ROC de 0.9011.

Aunque KNN había obtenido el mayor F1-Score durante la validación temporal interna, Random Forest presentó un mejor desempeño en el conjunto de prueba.

Según las matrices de confusión, Random Forest identificó correctamente 60 de los 90 días de demanda alta, mientras que KNN identificó 51.

No obstante, el conjunto de prueba presentó un desbalance importante, con aproximadamente 81.1 % de días de demanda alta. Una clasificación constante de demanda alta alcanzaría un Accuracy de 81.1 %, superior al de ambos modelos.

Por tanto, aunque Random Forest presentó una buena capacidad discriminativa según AUC-ROC, es necesario interpretar las métricas considerando el desbalance y el umbral de decisión utilizado.

Estos resultados evidencian la importancia de utilizar varias métricas y evaluar los modelos sobre períodos temporales diferentes.


In [ ]:

# ==========================================================
# FASE 3 - CELDA 24
# Consolidación y verificación de resultados
# ==========================================================

print("=" * 55)
print("RESUMEN FINAL - FASE 3")
print("=" * 55)

print("\n1. REGRESIÓN - CONJUNTO DE PRUEBA")
display(tabla_regresion_test.round(4))

print("\n2. CLASIFICACIÓN - CONJUNTO DE PRUEBA")
display(tabla_clasificacion_test.round(4))

print("\n3. CLUSTERING - COMPARACIÓN")
display(comparacion_clustering.round(4))

print("\n4. CONFIGURACIONES SELECCIONADAS")
print("Ridge alpha:", mejor_alpha)
print("Random Forest Regressor:", mejor_config_rf)
print("KNN k:", mejor_k)
print(
    "Random Forest Classifier:",
    int(mejor_config_rf_clf["n_estimators"]),
    mejor_config_rf_clf["max_depth"],
    int(mejor_config_rf_clf["min_samples_leaf"])
)
print("K-Means K:", mejor_k_cluster)

# Verificar archivos de resultados
archivos_esperados = [
    "comparacion_regresion.png",
    "comparacion_clasificacion.png",
    "kmeans_codo_silueta.png",
    "perfil_kmeans.png",
    "comparacion_perfiles_clusters.png",
    "importancia_variables_regresion.png",
    "evaluacion_final_regresion.png",
    "evaluacion_final_clasificacion.png"
]

print("\n5. VERIFICACIÓN DE GRÁFICOS")

archivos_faltantes = []

for nombre in archivos_esperados:
    ruta = RESULTADOS / nombre

    if ruta.exists():
        print("OK:", nombre)
    else:
        print("FALTA:", nombre)
        archivos_faltantes.append(nombre)

assert not archivos_faltantes, (
    f"Faltan archivos: {archivos_faltantes}"
)

assert len(tabla_regresion_test) == 2
assert len(tabla_clasificacion_test) == 2
assert len(comparacion_clustering) == 2

print("\nVERIFICACIÓN FINAL COMPLETADA.")



## 9. Conclusiones generales

### 9.1. Modelos de regresión

Se implementaron dos algoritmos para predecir la cantidad diaria de alquileres de bicicletas: Ridge Regression y Random Forest Regressor.

Durante la validación, Random Forest obtuvo el menor error de predicción. Sin embargo, en el conjunto final de prueba, Ridge presentó un mejor desempeño, con un RMSE de 1232.48, MAE de 938.57 y R² de 0.6215.

Estos resultados muestran que el rendimiento de los modelos puede variar entre períodos temporales y que una mayor complejidad algorítmica no garantiza una mejor capacidad de generalización.

### 9.2. Modelos de clasificación

Se implementaron K-Nearest Neighbors y Random Forest Classifier para identificar días de demanda alta y baja.

El umbral de clasificación fue de 3974 alquileres diarios, calculado exclusivamente a partir de la mediana del conjunto de entrenamiento.

En la validación temporal interna, KNN obtuvo el mejor F1-Score promedio, con un valor de 0.6428.

Sin embargo, en el conjunto final de prueba, Random Forest presentó mejores resultados, con un F1-Score de 0.7843 y AUC-ROC de 0.9011.

Se identificó un cambio temporal en la distribución de clases. El conjunto de validación externa contenía únicamente días de demanda alta, mientras que el conjunto de prueba presentó un 81.1 % de observaciones de esta categoría.

Esta situación demuestra la necesidad de interpretar las métricas considerando el desbalance y de utilizar estrategias de validación apropiadas para series temporales.

### 9.3. Aprendizaje no supervisado

Se aplicaron K-Means y Agglomerative Clustering para identificar patrones de días a partir de características meteorológicas y de calendario.

K-Means obtuvo el mayor coeficiente de silueta, con un valor de 0.1918 para dos clústeres. Agglomerative Clustering obtuvo una silueta de 0.1543.

Los perfiles de los grupos mostraron asociaciones entre temperaturas más elevadas, menor velocidad del viento y mayor demanda promedio de bicicletas.

No obstante, los coeficientes de silueta relativamente bajos indican que los grupos no presentan una separación claramente definida.

Por tanto, los resultados del clustering deben interpretarse como exploratorios.

### 9.4. Interpretabilidad

La importancia por permutación del modelo Random Forest Regressor identificó la humedad y la velocidad del viento como las características con mayor contribución predictiva durante la validación.

Estos resultados aportan información sobre el comportamiento del modelo, aunque no permiten establecer relaciones causales entre las variables meteorológicas y la demanda.

### 9.5. Limitaciones del estudio

Las principales limitaciones identificadas fueron:

1. El conjunto de datos contiene 731 observaciones correspondientes a dos años.
2. La demanda presenta cambios temporales que afectan el rendimiento de los modelos.
3. La clasificación presenta desbalance de clases en los períodos de validación y prueba.
4. Los modelos de regresión muestran errores relevantes en determinadas observaciones.
5. Los algoritmos de clustering presentan coeficientes de silueta bajos.
6. Los resultados corresponden al conjunto de datos analizado y no garantizan el mismo desempeño en otros sistemas de bicicletas.

### 9.6. Recomendaciones y trabajo futuro

Se recomienda explorar nuevas características temporales, como tendencias, variables rezagadas y patrones estacionales, siempre evitando utilizar información futura para realizar predicciones.

También sería conveniente ampliar la validación temporal, comparar modelos adicionales y analizar estrategias de calibración y selección del umbral de clasificación utilizando exclusivamente datos de entrenamiento y validación.

Para el aprendizaje no supervisado, podrían evaluarse otras representaciones de las variables, métodos de agrupamiento y técnicas de reducción de dimensionalidad.

### 9.7. Conclusión final

El proyecto permitió aplicar y comparar técnicas avanzadas de aprendizaje supervisado y no supervisado sobre datos reales de alquiler de bicicletas.

Los resultados evidenciaron que la evaluación de modelos debe considerar tanto su rendimiento predictivo como su capacidad de generalización, interpretabilidad y limitaciones metodológicas.

La comparación de regresión, clasificación y clustering permitió obtener una visión integral del problema y desarrollar competencias en preparación de datos, optimización de hiperparámetros, evaluación de modelos e interpretación de resultados.
